In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import string
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
users_code_location_df = pd.read_csv('/content/drive/MyDrive/thesis/kaggle/part2 feature-creation/2.1 feature creation/code_sections_extracts.csv')

users_code_location_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 24 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Username                     2000 non-null   object 
 1   Display_Name                 2000 non-null   object 
 2   Gender                       2000 non-null   object 
 3   notebook_url                 2000 non-null   object 
 4   code_location                2000 non-null   object 
 5   labels                       2000 non-null   object 
 6   top_labels                   2000 non-null   object 
 7   code_sections                2000 non-null   object 
 8   markdown_sections            2000 non-null   object 
 9   all_sections                 2000 non-null   object 
 10  only_code_in_code_sections   2000 non-null   object 
 11  number_of_lines              2000 non-null   float64
 12  names_set                    2000 non-null   object 
 13  num_of_sections   

In [4]:
users_code_location_df.head()

,Username,Display_Name,Gender,notebook_url,code_location,labels,top_labels,code_sections,markdown_sections,all_sections,...,token_count,variables_count,function_count,loop_count,condition_count,single_line_comment_density,function_density,loop_density,condition_density,comment_tokens_density
0,tchaye59,Jude TCHAYE,male,https://www.kaggle.com/code/tchaye59/jmarket-k...,/content/drive/MyDrive/thesis/notebooks/male/t...,"['Jane Street Market Prediction', 'Jane Street...",{'Jane Street Market Prediction'},['# This Python 3 environment comes with many ...,['### This notebook is only dedicated to submi...,['# This Python 3 environment comes with many ...,...,155,30,0,3,0,0.333333,0.000000,0.019355,0.000000,0.524272
1,iyara1,Riya,female,https://www.kaggle.com/code/iyara1/deepanalysi...,/content/drive/MyDrive/thesis/notebooks/female...,['House Prices - Advanced Regression Techniques'],{'House Prices - Advanced Regression Techniques'},"[""import numpy as np\nimport pandas as pd\nimp...",['**#Bivariate Analysis******'],"[""import numpy as np\nimport pandas as pd\nimp...",...,374,102,1,7,7,0.129825,0.002674,0.018717,0.018717,0.359568
2,sanjay7013,Sanjay M,male,https://www.kaggle.com/code/sanjay7013/credit-...,/content/drive/MyDrive/thesis/notebooks/male/s...,['Credit Card Fraud Detection'],{'Credit Card Fraud Detection'},['# This Python 3 environment comes with many ...,"['# Credit Card Fraud Detection', ""### DataSet...","['# Credit Card Fraud Detection', ""### DataSet...",...,206,65,0,2,0,0.166667,0.000000,0.009709,0.000000,0.386076
3,validmodel,Rashmi Margani,female,https://www.kaggle.com/code/validmodel/master-...,/content/drive/MyDrive/thesis/notebooks/female...,['Iris Species'],{'Iris Species'},['# This Python 3 environment comes with many ...,"[""# <h1 style='background:#f0c2c1; border:2; b...",['# This Python 3 environment comes with many ...,...,302,95,12,9,12,0.159420,0.039735,0.029801,0.039735,0.279314
4,rajeevnair676,Rajeev Nair,male,https://www.kaggle.com/code/rajeevnair676/nlp-...,/content/drive/MyDrive/thesis/notebooks/male/r...,"['IMDB Dataset of 50K Movie Reviews', 'Tweet S...",{'Natural Language Processing with Disaster Tw...,['#Importing NLTK package\nimport nltk\n\nimpo...,['# <center><b> NLP STARTERS - PART 1 <center/...,['# <center><b> NLP STARTERS - PART 1 <center/...,...,197,63,5,17,3,0.095652,0.025381,0.086294,0.015228,0.193370


In [5]:
def avg_var_name_length(names_set):
    return sum(len(name) for name in eval(names_set)) / len(eval(names_set)) if len(eval(names_set)) else 0

users_code_location_df['avg_var_name_length'] = users_code_location_df['names_set'].apply(avg_var_name_length)

In [6]:
def comment_to_code_ratio(code_sections, markdown_sections):
    code_lines = sum(len(section.split('\n')) for section in code_sections)
    comment_lines = sum(len(section.split('\n')) for section in markdown_sections)
    return comment_lines / code_lines if code_lines > 0 else 0

users_code_location_df['comment_to_code_ratio'] = users_code_location_df.apply(lambda row: comment_to_code_ratio(row['only_code_in_code_sections'], row['markdown_sections']), axis=1)

In [7]:
def avg_func_length(code_sections):
    func_lengths = []
    for section in eval(code_sections):
        lines = section.split('\n')
        func_start = None
        length = 0
        for i, line in enumerate(lines):
            if line.startswith('def '):
                if func_start is not None:
                    func_lengths.append(length)
                    length = 0
                func_start = i
            elif func_start is not None:
                length += 1
        if func_start is not None:
            func_lengths.append(length)
    return sum(func_lengths) / len(func_lengths) if func_lengths else 0

users_code_location_df['avg_func_length'] = users_code_location_df['code_sections'].apply(avg_func_length)

In [8]:
def code_to_markdown_ratio(code_sections, markdown_sections):
    return len(code_sections) / len(markdown_sections) if markdown_sections else 0

users_code_location_df['code_to_markdown_ratio'] = users_code_location_df.apply(lambda row: code_to_markdown_ratio(row['code_sections'], row['markdown_sections']), axis=1)

In [9]:
def avg_markdown_length(markdown_sections):
    return sum(len(section.split('\n')) for section in eval(markdown_sections)) / len(eval(markdown_sections)) if eval(markdown_sections) else 0

users_code_location_df['avg_markdown_lines_length'] = users_code_location_df['markdown_sections'].apply(avg_markdown_length)

In [10]:
def markdown_sentiment(markdown_sections):
    sia = SentimentIntensityAnalyzer()
    scores = []
    for section in eval(markdown_sections):
        score = sia.polarity_scores(section)['compound']
        scores.append(score)
    return sum(scores) / len(scores) if scores else 0

users_code_location_df['markdown_sentiment'] = users_code_location_df['markdown_sections'].apply(markdown_sentiment)

In [11]:
users_code_location_df.head()

,Username,Display_Name,Gender,notebook_url,code_location,labels,top_labels,code_sections,markdown_sections,all_sections,...,function_density,loop_density,condition_density,comment_tokens_density,avg_var_name_length,comment_to_code_ratio,avg_func_length,code_to_markdown_ratio,avg_markdown_lines_length,markdown_sentiment
0,tchaye59,Jude TCHAYE,male,https://www.kaggle.com/code/tchaye59/jmarket-k...,/content/drive/MyDrive/thesis/notebooks/male/t...,"['Jane Street Market Prediction', 'Jane Street...",{'Jane Street Market Prediction'},['# This Python 3 environment comes with many ...,['### This notebook is only dedicated to submi...,['# This Python 3 environment comes with many ...,...,0.000000,0.019355,0.000000,0.524272,6.333333,0.125708,0.000000,7.675676,1.000000,0.152933
1,iyara1,Riya,female,https://www.kaggle.com/code/iyara1/deepanalysi...,/content/drive/MyDrive/thesis/notebooks/female...,['House Prices - Advanced Regression Techniques'],{'House Prices - Advanced Regression Techniques'},"[""import numpy as np\nimport pandas as pd\nimp...",['**#Bivariate Analysis******'],"[""import numpy as np\nimport pandas as pd\nimp...",...,0.002674,0.018717,0.018717,0.359568,6.166667,0.002423,4.000000,398.903226,1.000000,0.000000
2,sanjay7013,Sanjay M,male,https://www.kaggle.com/code/sanjay7013/credit-...,/content/drive/MyDrive/thesis/notebooks/male/s...,['Credit Card Fraud Detection'],{'Credit Card Fraud Detection'},['# This Python 3 environment comes with many ...,"['# Credit Card Fraud Detection', ""### DataSet...","['# Credit Card Fraud Detection', ""### DataSet...",...,0.000000,0.009709,0.000000,0.386076,6.846154,0.542884,0.000000,1.788180,1.235294,0.004424
3,validmodel,Rashmi Margani,female,https://www.kaggle.com/code/validmodel/master-...,/content/drive/MyDrive/thesis/notebooks/female...,['Iris Species'],{'Iris Species'},['# This Python 3 environment comes with many ...,"[""# <h1 style='background:#f0c2c1; border:2; b...",['# This Python 3 environment comes with many ...,...,0.039735,0.029801,0.039735,0.279314,7.084211,0.755798,6.333333,1.261214,8.736842,0.414937
4,rajeevnair676,Rajeev Nair,male,https://www.kaggle.com/code/rajeevnair676/nlp-...,/content/drive/MyDrive/thesis/notebooks/male/r...,"['IMDB Dataset of 50K Movie Reviews', 'Tweet S...",{'Natural Language Processing with Disaster Tw...,['#Importing NLTK package\nimport nltk\n\nimpo...,['# <center><b> NLP STARTERS - PART 1 <center/...,['# <center><b> NLP STARTERS - PART 1 <center/...,...,0.025381,0.086294,0.015228,0.193370,6.698413,2.530189,11.400000,0.377135,5.000000,0.407079


In [12]:
users_code_location_df.to_csv('/content/drive/MyDrive/thesis/kaggle/part2 feature-creation/2.3 feature engineering/kaggle_df.csv', index=False)